# **Comparación de modelos de ML con Redes Nuronales usando Housing y German Credit**
Aprendizaje Profundo
- Daniel Velasco González

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

## **Baseline models**

### **German Credit Dataset**
#### **Model**: DecisionTree Classifier

In [2]:
# @title
df = pd.read_csv('https://raw.githubusercontent.com/camilousa/datasets/refs/heads/master/german_credit_data.csv')

# 1. Select 'Age' and 'Credit amount' for X
X = df[['Age', 'Credit amount']]

# 2. Select 'Risk' for y
y = df['Risk']

# 3. Encode the 'Risk' variable
le = LabelEncoder()
y = le.fit_transform(y)

# 4. Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Instantiate Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(
    random_state=42,
    max_depth=2
) # Added random_state for reproducibility

# Train the Decision Tree Classifier
dt_classifier.fit(X_train, y_train)

# Make predictions on the training set
y_train_pred_dt = dt_classifier.predict(X_train)

# Make predictions on the test set
y_test_pred_dt = dt_classifier.predict(X_test)

# Calculate metrics for training set
accuracy_train_dt = accuracy_score(y_train, y_train_pred_dt)
precision_train_dt = precision_score(y_train, y_train_pred_dt)
recall_train_dt = recall_score(y_train, y_train_pred_dt)
f1_train_dt = f1_score(y_train, y_train_pred_dt)

# Calculate metrics for test set
accuracy_test_dt = accuracy_score(y_test, y_test_pred_dt)
precision_test_dt = precision_score(y_test, y_test_pred_dt)
recall_test_dt = recall_score(y_test, y_test_pred_dt)
f1_test_dt = f1_score(y_test, y_test_pred_dt)

# Create a DataFrame for Decision Tree results
dt_results = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training Set': [accuracy_train_dt, precision_train_dt, recall_train_dt, f1_train_dt],
    'Test Set': [accuracy_test_dt, precision_test_dt, recall_test_dt, f1_test_dt]
}
dt_results_df = pd.DataFrame(dt_results)

print("\nDecision Tree Classifier Performance:")
dt_results_df.head()


Decision Tree Classifier Performance:


,Metric,Training Set,Test Set
0,Accuracy,0.718571,0.703333
1,Precision,0.717456,0.702703
2,Recall,0.987780,0.995215
3,F1-Score,0.831191,0.823762


### **Housing Dataset**
#### **Model**: DecisionTree Regressor

In [19]:
# @title
# 1. Cargar el dataset de housing
df = pd.read_csv('https://raw.githubusercontent.com/camilousa/datasets/refs/heads/master/housing.csv')

# 2. Seleccionar variables X (usaremos ingresos medios y antigüedad de la vivienda para mantenerlo simple)
X = df[['median_income', 'housing_median_age']]

# 3. Seleccionar 'median_house_value' como variable objetivo y (lo que queremos predecir)
y = df['median_house_value']

# 4. Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 5. Instanciar el modelo (Regresión en lugar de Clasificación)
dt_regressor = DecisionTreeRegressor(
    random_state=42,
    max_depth=4
)

# 6. Entrenar el Árbol de Decisión
dt_regressor.fit(X_train, y_train)

# 7. Hacer predicciones en ambos conjuntos
y_train_pred_dt = dt_regressor.predict(X_train)
y_test_pred_dt = dt_regressor.predict(X_test)

# 8. Calcular métricas para el conjunto de entrenamiento
r2_train_dt = r2_score(y_train, y_train_pred_dt)
mae_train_dt = mean_absolute_error(y_train, y_train_pred_dt)
rmse_train_dt = np.sqrt(mean_squared_error(y_train, y_train_pred_dt))

# 9. Calcular métricas para el conjunto de prueba
r2_test_dt = r2_score(y_test, y_test_pred_dt)
mae_test_dt = mean_absolute_error(y_test, y_test_pred_dt)
rmse_test_dt = np.sqrt(mean_squared_error(y_test, y_test_pred_dt))

# 10. Crear un DataFrame para ver los resultados
dt_results = {
    'Métrica': ['R2 (R-cuadrado)', 'MAE (Error Absoluto Medio)', 'RMSE (Raíz del Error Cuadrático)'],
    'Entrenamiento': [r2_train_dt, mae_train_dt, rmse_train_dt],
    'Prueba': [r2_test_dt, mae_test_dt, rmse_test_dt]
}
dt_results_df = pd.DataFrame(dt_results)

print("\nRendimiento del Árbol de Decisión (Regresión):")
dt_results_df.head()


Rendimiento del Árbol de Decisión (Regresión):


,Métrica,Entrenamiento,Prueba
0,R2 (R-cuadrado),0.527364,0.511914
1,MAE (Error Absoluto Medio),58968.835367,59523.180128
2,RMSE (Raíz del Error Cuadrático),79573.174559,80039.551117


## **Redes Neuronales**

### **German Credit Dataset**

In [25]:
# @title
# 1. Cargar el dataset
df = pd.read_csv('https://raw.githubusercontent.com/camilousa/datasets/refs/heads/master/german_credit_data.csv')

# 2. Seleccionar X e y
X = df[['Age', 'Credit amount']]
y = df['Risk']

# 3. Codificar la variable 'Risk' (0 para 'bad', 1 para 'good')
le = LabelEncoder()
y = le.fit_transform(y)

# 4. Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 5. Escalar los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Construir la Red Neuronal
nn_classifier = Sequential([
    Dense(8, activation='relu', input_shape=(2,)),
    Dense(4, activation='relu'),
    Dense(2, activation='softmax')
])

# 7. Compilar el modelo
nn_classifier.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 8. Entrenar la Red Neuronal
print("Entrenando la red neuronal... (esto puede tomar unos segundos)")
nn_classifier.fit(X_train_scaled, y_train, epochs=100, batch_size=32, verbose=0)

# 9. Hacer predicciones (Ahora devuelve 2 probabilidades por fila)
y_train_prob = nn_classifier.predict(X_train_scaled, verbose=0)
y_test_prob = nn_classifier.predict(X_test_scaled, verbose=0)

# ¡EL CAMBIO ESTÁ AQUÍ! np.argmax selecciona la columna (clase 0 o 1) con la probabilidad mayor
y_train_pred_nn = np.argmax(y_train_prob, axis=1)
y_test_pred_nn = np.argmax(y_test_prob, axis=1)

# 10. Calcular métricas
def get_metrics(y_true, y_pred):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    ]

nn_results = {
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Entrenamiento': get_metrics(y_train, y_train_pred_nn),
    'Prueba': get_metrics(y_test, y_test_pred_nn)
}

nn_results_df = pd.DataFrame(nn_results)

print("\nRendimiento de la Red Neuronal (Softmax - 2 Neuronas):")
nn_results_df.head()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entrenando la red neuronal... (esto puede tomar unos segundos)

Rendimiento de la Red Neuronal (Softmax - 2 Neuronas):


,Métrica,Entrenamiento,Prueba
0,Accuracy,0.712857,0.713333
1,Precision,0.718373,0.708475
2,Recall,0.971487,1.000000
3,F1-Score,0.825974,0.829365


In [35]:
# 1. Cargar el dataset
df = pd.read_csv('https://raw.githubusercontent.com/camilousa/datasets/refs/heads/master/housing.csv')

# 2. Limpieza de Outliers (Lo que ya hicimos)
df = df[df['median_house_value'] < 500000]
Q1 = df['median_income'].quantile(0.25)
Q3 = df['median_income'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR
df = df[df['median_income'] <= limite_superior]

# =====================================================================
# --- NUEVO: PRE-PROCESAMIENTO AVANZADO ---

# a. Rellenar valores nulos: La columna 'total_bedrooms' tiene datos faltantes.
# Los rellenamos con la mediana para no perder esas filas.
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

# b. Convertir texto a números: 'ocean_proximity' es texto.
# Usamos "One-Hot Encoding" para crear columnas de 0s y 1s (Ej: ¿Está cerca del mar? 1=Sí, 0=No)
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
# =====================================================================

# 3. Seleccionar X e y (¡AHORA USAMOS TODAS LAS COLUMNAS!)
X = df.drop('median_house_value', axis=1) # X es todo excepto el precio
y = df['median_house_value']

# 4. Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 5. Escalar los datos (Aún más importante ahora que hay variables grandes como 'population')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Construir la Red Neuronal (Más robusta para procesar más datos)
# Nota: input_shape ahora toma dinámicamente el número de columnas de X
nn_regressor = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])

# 7. Compilar el modelo
nn_regressor.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 8. Entrenar la Red Neuronal (Subimos un poco las epochs)
print(f"Entrenando la red neuronal con {X_train_scaled.shape[1]} variables... (puede tardar un poco más)")
nn_regressor.fit(X_train_scaled, y_train, epochs=150, batch_size=32, verbose=0)

# 9. Hacer predicciones
y_train_pred_nn = nn_regressor.predict(X_train_scaled, verbose=0).flatten()
y_test_pred_nn = nn_regressor.predict(X_test_scaled, verbose=0).flatten()

# 10. Calcular métricas
def get_regression_metrics(y_true, y_pred):
    return [
        r2_score(y_true, y_pred),
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred))
    ]

nn_results = {
    'Métrica': ['R2 (R-cuadrado)', 'MAE (Error Absoluto)', 'RMSE'],
    'Entrenamiento': get_regression_metrics(y_train, y_train_pred_nn),
    'Prueba': get_regression_metrics(y_test, y_test_pred_nn)
}

nn_results_df = pd.DataFrame(nn_results)

print("\nRendimiento de la Red Neuronal (CON TODAS LAS VARIABLES):")
nn_results_df.head()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entrenando la red neuronal con 12 variables... (puede tardar un poco más)

Rendimiento de la Red Neuronal (CON TODAS LAS VARIABLES):


,Métrica,Entrenamiento,Prueba
0,R2 (R-cuadrado),0.658555,0.665497
1,MAE (Error Absoluto),38784.879372,39267.309386
2,RMSE,54623.883287,54654.610448
